In [1]:
from workloads import vgg16, simple_net, resnet18
from workloads.common import *

from roofline.roofline_model import *

# net = vgg16.network(resolutions['original'])
# save_path = 'files/vgg16'

net = resnet18.network(resolutions['original'])
save_path = 'files/simple_net'

board_part = 'zcu102'
double_buff=True
num_hp = 1
t_factor = 2
bits = 8
buswidth = 512

In [2]:
sol_ls = []
layer_cycles = []
for layer_idx in net.keys():
    print("cuttent layer: ", layer_idx)
    layer_meta = net[layer_idx]

    # search design for a layer
    pair_ls, comp_bnd, bw_bnd, solution = DSE_layer(\
        layer_meta, board_part, layer_idx, bits, buswidth, t_factor, save_path, \
        double_buff=double_buff, num_hp = num_hp)

    sol_ls.append(solution[1])
    R = layer_meta['niy']; C = layer_meta['nix']; N = layer_meta['nif']; M = layer_meta['nof']; K = layer_meta['kernel']
    S = 1
    Tr = solution[1][0];Tc = solution[1][1];Tn = solution[1][2];Tm = solution[1][3]
    layer_cycles.append(exec_cycles(R, C, N, M, Tr, Tc, Tn, Tm, S, K))

# averaged to final solution
avg_Tr = list(zip(*sol_ls))[0]; avg_Tr = statistics.mean(avg_Tr)
avg_Tc = list(zip(*sol_ls))[1]; avg_Tc = statistics.mean(avg_Tc)
avg_Tn = list(zip(*sol_ls))[2]; avg_Tn = statistics.mean(avg_Tn)
avg_Tm = list(zip(*sol_ls))[3]; avg_Tm = statistics.mean(avg_Tm)

all_layer_cycle = 0
for layer_idx in net.keys():
    layer_meta = net[layer_idx]
    # count each layer time
    R = layer_meta['niy']; C = layer_meta['nix']; N = layer_meta['nif']; M = layer_meta['nof']; K = layer_meta['kernel']
    S = 1
    all_layer_cycle += exec_cycles(R, C, N, M, avg_Tr, avg_Tc, avg_Tn, avg_Tm, S, K)

print("averaged Tr = {}, Tc = {}, Tn = {}, Tm = {}".format(avg_Tr, avg_Tc, avg_Tn, avg_Tm))
print("sum of each layer exec_cycles: ", sum(layer_cycles))
print("sum of uniformed layer exec_cycles: ", all_layer_cycle)
print("latency(ms): ", all_layer_cycle/(200*(10**6)) * (10**3))

  1%|          | 7869/1048576 [00:00<00:13, 78688.39it/s]

cuttent layer:  conv1


  1%|          | 35454/4194304 [00:00<00:23, 176196.52it/s]

cuttent layer:  conv2_1


  1%|          | 35220/4194304 [00:00<00:23, 175737.99it/s]

...

cuttent layer:  conv9_2


 19%|█▉        | 157228/835584 [00:00<00:00, 1572262.76it/s]

cuttent layer:  fc1


100%|██████████| 835584/835584 [00:00<00:00, 1849721.55it/s]


averaged Tr = 24.944444444444443, Tc = 26.72222222222222, Tn = 22.22222222222222, Tm = 38.333333333333336
sum of each layer exec_cycles:  23172225.0
sum of uniformed layer exec_cycles:  49717974.506172836
latency(ms):  248.58987253086417


In [3]:
Tr = 24; Tc =26; Tn = 22; Tm = 38; K=3; S=1

In [4]:
all_layer_cycle = 0
for layer_idx in net.keys():
    layer_meta = net[layer_idx]
    # count each layer time
    R = layer_meta['niy']; C = layer_meta['nix']; N = layer_meta['nif']; M = layer_meta['nof']; K = layer_meta['kernel']
    S = layer_meta['stride'] if 'stride' in layer_meta.keys() else 1
    each_layer_cycle = exec_cycles(R, C, N, M, Tr, Tc, Tn, Tm, S, K)
    print(each_layer_cycle/(200*(10**6)) * (10**3))
    all_layer_cycle += each_layer_cycle

print("averaged Tr = {}, Tc = {}, Tn = {}, Tm = {}".format(Tr, Tc, Tn, Tm))
print("sum of each layer exec_cycles: ", sum(layer_cycles))
print("sum of uniformed layer exec_cycles: ", all_layer_cycle)
print("latency(ms): ", all_layer_cycle/(200*(10**6)) * (10**3))

39.6913
6.2046
6.2046
6.2046
15.7734
7.3332
7.3332
7.3332
16.36551
11.31984
11.31984
11.31984
28.919520000000002
11.27616
11.27616
11.27616
11.27616
28.751459999999998
averaged Tr = 24, Tc = 26, Tn = 22, Tm = 38
sum of each layer exec_cycles:  23172225.0
sum of uniformed layer exec_cycles:  49835750.0
latency(ms):  249.17875


In [8]:
beta_in = Tn * (S*Tr + K-S) * (S*Tc + K-S)
beta_wgt = Tn * Tm * K * K
beta_out = Tm * Tr * Tc

In [9]:
bram_usage(beta_in, beta_wgt, beta_out, Tn, Tm, bits = bits, double_buff=False)

450

In [10]:
dsp_usage(Tn, Tm, bits=bits)

418

In [ ]:
39.6913
6.2046
6.2046
6.2046
15.7734
7.3332
7.3332
7.3332
16.36551
11.31984
11.31984
11.31984
28.919520000000002
11.27616
11.27616
11.27616
11.27616
28.751459999999998

In [14]:
11.27616*4

45.10464